<a href="https://colab.research.google.com/github/AliRaddman/divar-ml-project/blob/main/notebooks/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" align="right">

# نوت‌بوک آماده‌سازی و تمیزکاری داده‌ها

این نوت‌بوک نسخه‌ی نهایی و ادغام‌شده‌ی مرحله‌ی preprocessing پروژه است.

در این فایل، خروجی‌های کاری اعضای تیم از نوت‌بوک‌های جداگانه ادغام می‌شوند و دیتاست خام دیوار به یک نسخه‌ی تمیز و قابل استفاده برای تحلیل آماری، نقشه، خوشه‌بندی و مدل‌سازی تبدیل می‌شود.

مراحل اصلی این نوت‌بوک:

1. راه‌اندازی محیط و نصب کتابخانه‌ها
2. بارگذاری دیتای خام
3. اصلاح نوع داده‌ها و مدیریت مقادیر گم‌شده
4. ساخت ستون‌های مربوط به قیمت، متراژ و قیمت هدف
5. ساخت flagهای اعتبارسنجی برای تحلیل قیمت
6. پاکسازی داده‌های جغرافیایی
7. تبدیل مختصات جغرافیایی به UTM
8. ذخیره خروجی نهایی preprocessing

</div>

<div dir="rtl" align="right">

## 1. راه‌اندازی محیط و import کتابخانه‌ها

در این مرحله Google Drive به Colab متصل می‌شود و کتابخانه‌های موردنیاز برای preprocessing، کار با فایل‌های parquet، تبدیل تاریخ شمسی و تبدیل مختصات به UTM آماده می‌شوند.

</div>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q jdatetime utm pyarrow

import os
import pandas as pd
import numpy as np
import jdatetime
import utm

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

print("Setup completed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup completed.


<div dir="rtl" align="right">

## 2. بارگذاری دیتای خام

در این مرحله دیتاست خام آگهی‌های املاک دیوار از Google Drive خوانده می‌شود.

این فایل نقطه شروع preprocessing است و در مراحل بعدی، نوع داده‌ها، مقادیر گم‌شده، ستون‌های قیمت و مختصات جغرافیایی روی آن پردازش می‌شوند.

</div>

In [2]:
raw_data_path = "/content/drive/MyDrive/Divar Dataset/Divar.csv"

df = pd.read_csv(raw_data_path)

print("Shape:")
print(df.shape)

print("\nNumber of columns:")
print(len(df.columns))

df.head()

/tmp/ipykernel_61213/2399571182.py:3: DtypeWarning: Columns (11,27,29,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_data_path)


Shape:
(1000000, 61)

Number of columns:
61


,Unnamed: 0,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,rent_mode,rent_value,rent_to_single,rent_type,price_mode,price_value,credit_mode,credit_value,rent_credit_transform,transformable_price,transformable_credit,transformed_credit,transformable_rent,transformed_rent,land_size,building_size,deed_type,has_business_deed,floor,rooms_count,total_floors_count,unit_per_floor,has_balcony,has_elevator,has_warehouse,has_parking,construction_year,is_rebuilt,has_water,has_warm_water_provider,has_electricity,has_gas,has_heating_system,has_cooling_system,has_restroom,has_security_guard,has_barbecue,building_direction,has_pool,has_jacuzzi,has_sauna,floor_material,property_type,regular_person_capacity,extra_person_capacity,cost_per_extra_person,rent_price_on_regular_days,rent_price_on_special_days,rent_price_at_weekends,location_latitude,location_longitude,location_radius
0,0,temporary-rent,villa,karaj,mehrshahr,2024-08-01 00:00:00,مشاور املاک,۵۰۰متر\n۲۰۰متر بنا دوبلکس\n۳خواب\nاستخر آبگرم ...,باغ ویلا اجاره روزانه استخر داخل لشکرآباد سهیلیه,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0,NaN,NaN,NaN,سه,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,6,350000.0,1500000.0,3.500000e+09,3500000.0,35.811684,50.936600,500.0
1,1,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,NaN,NaN,NaN,مقطوع,8.500000e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,NaN,NaN,3,یک,NaN,NaN,NaN,True,True,True,۱۳۸۴,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0
2,2,residential-rent,apartment-rent,tehran,tohid,2024-10-01 00:00:00,NaN,تخلیه پایان ماه,آپارتمان ۳ خوابه ۱۳۲ متر,مقطوع,26000000.0,NaN,NaN,NaN,NaN,مقطوع,750000000.0,False,False,750000000.0,NaN,26000000.0,NaN,NaN,132.0,NaN,NaN,3,سه,NaN,NaN,NaN,True,True,True,۱۴۰۱,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.703865,51.373459,NaN
3,3,commercial-rent,office-rent,tehran,elahiyeh,2024-06-01 00:00:00,NaN,فرشته تاپ لوکیشن\n۹۰ متر موقعیت اداری\nیک اتاق...,فرشته ۹۰ متر دفتر کار مدرن موقعیت اداری,مقطوع,95000000.0,NaN,NaN,NaN,NaN,مقطوع,950000000.0,False,False,950000000.0,NaN,95000000.0,NaN,NaN,90.0,NaN,NaN,4,یک,NaN,NaN,NaN,True,False,True,۱۴۰۰,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,residential-sell,apartment-sell,mashhad,emamreza,2024-05-01 00:00:00,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,NaN,NaN,NaN,NaN,مقطوع,5.750000e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,115.0,single_page,NaN,4,دو,6,NaN,true,True,True,True,۱۴۰۳,NaN,NaN,package,NaN,NaN,shoofaj,air_conditioner,squat_seat,NaN,NaN,north,NaN,NaN,NaN,ceramic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<div dir="rtl" align="right">

## 3. اصلاح نوع داده‌ها و مدیریت مقادیر گم‌شده

در این بخش ستون‌های عددی، دسته‌ای، بولین و تاریخی اصلاح می‌شوند.

همچنین مقادیر گم‌شده بر اساس منطق هر ستون مدیریت می‌شوند.
برخی NaNها به دلیل ماهیت ساختاری ستون‌ها حفظ می‌شوند و برخی دیگر با روش‌هایی مثل میانه‌ی کل یا میانه‌ی هر دسته پر می‌شوند.

خروجی این مرحله:

`cleaned_step1.parquet`

</div>


<div dir="rtl" align="right">

### 3.1 اصلاح نوع داده‌ها

در این مرحله نوع داده‌ی ستون‌های اصلی اصلاح می‌شود.

ستون‌های عددی مثل `floor`، `total_floors_count`، `rooms_count`، `unit_per_floor` و `construction_year` به عدد تبدیل می‌شوند.

ستون‌های بولین مثل امکانات ملک و flagهای قیمتی به نوع `boolean` تبدیل می‌شوند.

همچنین مقدار `unselect` به `NaN` تبدیل می‌شود و ستون‌های دسته‌ای به نوع `category` تغییر می‌کنند.

</div>

In [3]:
# ستون‌های مربوط به طبقه را عددی می‌کنیم
for col in ["floor", "total_floors_count"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# تعداد اتاق‌ها در دیتا به‌صورت متنی آمده بود
rooms_map = {
    "بدون اتاق": 0,
    "یک": 1,
    "دو": 2,
    "سه": 3,
    "چهار": 4,
    "پنج یا بیشتر": 5,
}

df["rooms_count"] = df["rooms_count"].map(rooms_map)


# تعداد واحد در هر طبقه را عددی می‌کنیم
df["unit_per_floor"] = df["unit_per_floor"].replace("more_than_8", 9)
df["unit_per_floor"] = pd.to_numeric(df["unit_per_floor"], errors="coerce")


# ارقام فارسی سال ساخت را به انگلیسی تبدیل می‌کنیم
def fa_to_en_digits(text):
    if pd.isna(text):
        return text

    fa_digits = "۰۱۲۳۴۵۶۷۸۹"
    en_digits = "0123456789"
    translation_table = str.maketrans(fa_digits, en_digits)

    return str(text).translate(translation_table)


df["construction_year"] = df["construction_year"].apply(fa_to_en_digits)
df["construction_year"] = df["construction_year"].replace("قبل از 1370", 1365)
df["construction_year"] = pd.to_numeric(df["construction_year"], errors="coerce")


# این ستون‌ها واقعاً بولین هستند
clean_bool_cols = [
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "has_security_guard",
    "has_barbecue",
    "has_pool",
    "has_jacuzzi",
    "has_sauna",
    "has_business_deed",
    "is_rebuilt",
    "transformable_price",
    "rent_credit_transform",
]

# بالکن به‌صورت متن ذخیره شده بود، اول مقدارهایش را درست می‌کنیم
df["has_balcony"] = df["has_balcony"].replace({
    "true": True,
    "false": False,
    "unselect": np.nan,
})

all_bool_cols = clean_bool_cols + ["has_balcony"]

for col in all_bool_cols:
    df[col] = df[col].astype("boolean")


# مقدار unselect یعنی انتخاب نشده، پس به NaN تبدیلش می‌کنیم
df = df.replace("unselect", np.nan)


# ستون‌های دسته‌ای را category می‌کنیم که هم سبک‌تر باشند هم معنی‌شان مشخص‌تر شود
category_cols = [
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "neighborhood_slug",
    "user_type",
    "rent_mode",
    "rent_type",
    "price_mode",
    "credit_mode",
    "deed_type",
    "building_direction",
    "floor_material",
    "property_type",
    "has_warm_water_provider",
    "has_heating_system",
    "has_cooling_system",
    "has_restroom",
]

for col in category_cols:
    df[col] = df[col].astype("category")


# این ستون فقط index قبلی فایل بوده و لازم نیست
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])


print("Step 3.1 completed.")
print("Shape:", df.shape)
print("\nDtype summary:")
print(df.dtypes.value_counts())

Step 3.1 completed.
Shape: (1000000, 60)

Dtype summary:
float64     22
boolean     13
object       8
category     3
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


<div dir="rtl" align="right">

### 3.2 تبدیل تاریخ و حذف ستون‌های غیرضروری

در این مرحله از ستون `created_at_month` دو ستون شمسی می‌سازیم.

منطق تبدیل تاریخ مطابق نوت‌بوک کاری علی است، اما برای سریع‌تر شدن اجرا، تبدیل فقط روی مقدارهای یکتای `created_at_month` انجام می‌شود و بعد نتیجه روی کل دیتاست map می‌شود.

همچنین چند ستون که برای ادامه تحلیل لازم نیستند یا داده‌ی قابل اتکایی ندارند، از دیتاست حذف می‌شوند.

</div>

In [4]:
# اول created_at_month را به datetime تبدیل می‌کنیم
df["created_at_month"] = pd.to_datetime(df["created_at_month"], errors="coerce")


# تبدیل تاریخ میلادی به سال-ماه شمسی
def to_shamsi_month(date):
    if pd.isna(date):
        return np.nan

    jd = jdatetime.date.fromgregorian(
        year=date.year,
        month=date.month,
        day=date.day,
    )

    return f"{jd.year}-{jd.month:02d}"


# اسم ماه‌های شمسی برای نسخه‌ی خواناتر
month_names = {
    1: "فروردین",
    2: "اردیبهشت",
    3: "خرداد",
    4: "تیر",
    5: "مرداد",
    6: "شهریور",
    7: "مهر",
    8: "آبان",
    9: "آذر",
    10: "دی",
    11: "بهمن",
    12: "اسفند",
}


def to_shamsi_readable(date):
    if pd.isna(date):
        return np.nan

    jd = jdatetime.date.fromgregorian(
        year=date.year,
        month=date.month,
        day=date.day,
    )

    return f"{month_names[jd.month]} {jd.year}"


# چون created_at_month فقط چند مقدار یکتا دارد، تبدیل را فقط برای همان‌ها انجام می‌دهیم
unique_months = df["created_at_month"].dropna().unique()

shamsi_month_map = {
    month: to_shamsi_month(month)
    for month in unique_months
}

shamsi_readable_map = {
    month: to_shamsi_readable(month)
    for month in unique_months
}


df["created_at_shamsi"] = df["created_at_month"].map(shamsi_month_map)
df["created_at_shamsi_readable"] = df["created_at_month"].map(shamsi_readable_map)


# ستون‌هایی که برای ادامه کار لازم نیستند را حذف می‌کنیم
cols_to_drop = [
    "rent_to_single",
    "cost_per_extra_person",
    "rent_price_on_special_days",
    "rent_price_at_weekends",
    "rent_price_on_regular_days",
    "extra_person_capacity",
    "has_water",
    "has_electricity",
    "has_gas",
]

df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])


print("Step 3.2 completed.")
print("Shape:", df.shape)

df[
    [
        "created_at_month",
        "created_at_shamsi",
        "created_at_shamsi_readable",
    ]
].head()

Step 3.2 completed.
Shape: (1000000, 53)


,created_at_month,created_at_shamsi,created_at_shamsi_readable
0,2024-08-01,1403-05,مرداد 1403
1,2024-05-01,1403-02,اردیبهشت 1403
2,2024-10-01,1403-07,مهر 1403
3,2024-06-01,1403-03,خرداد 1403
4,2024-05-01,1403-02,اردیبهشت 1403


<div dir="rtl" align="right">

### 3.3 مدیریت مقادیر گم‌شده

در این مرحله مقادیر گم‌شده‌ی چند ستون مهم مدیریت می‌شوند.

برای ستون‌های امکانات، مقدار خالی به معنی ثبت‌نشدن آن امکان در آگهی در نظر گرفته شده و با `False` پر می‌شود.

مقدار گم‌شده‌ی `construction_year` با میانه‌ی کل پر می‌شود.

مقادیر گم‌شده‌ی `rooms_count` و `building_size` با میانه‌ی هر `cat3_slug` پر می‌شوند. اگر برای یک دسته میانه قابل محاسبه نباشد، مقدار خالی همان‌طور باقی می‌ماند.

در پایان، ردیف‌هایی که ستون‌های کلیدی `title`، `cat3_slug` یا `city_slug` را ندارند حذف می‌شوند.

</div>

<div dir="rtl" align="right">

### مدیریت مقدارهای گم‌شده‌ی سال ساخت

در بررسی داده مشخص شد مقدارهای گم‌شده‌ی ستون `construction_year` تصادفی نیستند و در بعضی دسته‌ها به‌صورت ساختاری وجود دارند.  
بنابراین ستون اصلی `construction_year` را با میانه پر نمی‌کنیم تا توزیع واقعی سال ساخت حفظ شود.

برای استفاده‌های بعدی، دو ستون کمکی ساخته می‌شود:

`construction_year_was_missing`

مشخص می‌کند آیا مقدار سال ساخت در داده‌ی خام خالی بوده یا نه.

`construction_year_imputed`

نسخه‌ای از سال ساخت است که مقدارهای خالی آن با میانه‌ی کل پر شده‌اند و در صورت نیاز می‌تواند برای مدل‌سازی استفاده شود.

</div>

In [5]:
# اگر امکانات خالی باشند، یعنی در آگهی ثبت نشده‌اند
facility_cols = [
    "has_parking",
    "has_warehouse",
    "has_balcony",
    "has_elevator",
    "has_security_guard",
    "has_sauna",
    "has_jacuzzi",
    "has_pool",
    "has_barbecue",
]

for col in facility_cols:
    df[col] = df[col].fillna(False).astype("boolean")


# برای is_rebuilt هم مقدار خالی را False در نظر می‌گیریم
df["is_rebuilt"] = df["is_rebuilt"].fillna(False).astype("boolean")


# سال ساخت را در ستون اصلی دست‌کاری نمی‌کنیم
# چون مقدارهای خالی آن در بعضی دسته‌ها ساختاری هستند
df["construction_year_was_missing"] = df["construction_year"].isna()

construction_year_median = df["construction_year"].median()

df["construction_year_imputed"] = df["construction_year"].fillna(
    construction_year_median
)


# تعداد اتاق و متراژ بنا را با میانه هر cat3_slug پر می‌کنیم
for col in ["rooms_count", "building_size"]:
    df[col] = df.groupby("cat3_slug", observed=True)[col].transform(
        lambda x: x.fillna(x.median())
    )


# ردیف‌هایی که ستون‌های کلیدی ندارند برای ادامه کار قابل استفاده نیستند
required_cols = ["title", "cat3_slug", "city_slug"]

rows_before = len(df)
df = df.dropna(subset=required_cols)
rows_after = len(df)

print("Step 3.3 completed.")
print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows dropped:", rows_before - rows_after)
print("Shape:", df.shape)

print("\nConstruction year median:")
print(construction_year_median)

print("\nMissing values in selected columns:")
print(df[[
    "construction_year",
    "construction_year_was_missing",
    "construction_year_imputed",
    "rooms_count",
    "building_size",
    "title",
    "cat3_slug",
    "city_slug",
]].isna().sum())

print("\nRows with missing construction_year:")
print(df["construction_year_was_missing"].sum())

Step 3.3 completed.
Rows before: 1000000
Rows after: 999943
Rows dropped: 57
Shape: (999943, 55)

Construction year median:
1395.0

Missing values in selected columns:
construction_year                184164
construction_year_was_missing         0
construction_year_imputed             0
rooms_count                       19402
building_size                         0
title                                 0
cat3_slug                             0
city_slug                             0
dtype: int64

Rows with missing construction_year:
184164


<div dir="rtl" align="right">

### 3.4 ذخیره خروجی مرحله اول preprocessing

در این مرحله خروجی تمیزکاری اولیه ذخیره می‌شود.

این فایل شامل اصلاح نوع داده‌ها، تبدیل تاریخ شمسی، مدیریت بخشی از مقادیر گم‌شده و حذف ردیف‌های دارای ستون‌های کلیدی ناقص است.

خروجی این مرحله ورودی مرحله بعدی، یعنی ساخت ستون‌های قیمت و flagهای اعتبارسنجی، خواهد بود.

</div>

In [6]:
# مسیر پوشه خروجی‌های تیم
team_data_path = "/content/drive/MyDrive/divar_project"

# اگر پوشه وجود نداشت، ساخته می‌شود
os.makedirs(team_data_path, exist_ok=True)

step1_output_path = os.path.join(team_data_path, "cleaned_step1.parquet")

df.to_parquet(step1_output_path, index=False)

print("Step 1 output saved.")
print("Path:", step1_output_path)
print("Shape:", df.shape)

Step 1 output saved.
Path: /content/drive/MyDrive/divar_project/cleaned_step1.parquet
Shape: (999943, 55)


<div dir="rtl" align="right">

## 4. preprocessing بنیامین: قیمت، قیمت واحد و flagهای اعتبارسنجی

در این بخش، کدهای مربوط به بنیامین از نوت‌بوک `prep_ben.ipynb` وارد می‌شود.

ورودی این مرحله، خروجی مرحله علی یعنی فایل `cleaned_step1.parquet` است.

در این مرحله هیچ ردیفی حذف نمی‌شود و فقط ستون‌های کمکی مربوط به قیمت، متراژ، قیمت هدف، داده‌های پرت و مختصات جغرافیایی ساخته می‌شوند.

</div>

In [7]:
# ورودی بخش بنیامین، خروجی مرحله علی است
INPUT_PATH_BEN = "/content/drive/MyDrive/divar_project/cleaned_step1.parquet"

print("Input path:", INPUT_PATH_BEN)
print("Exists:", os.path.exists(INPUT_PATH_BEN))

if not os.path.exists(INPUT_PATH_BEN):
    raise FileNotFoundError("cleaned_step1.parquet پیدا نشد. اول مرحله علی را ذخیره کن.")

df_base_v2 = pd.read_parquet(INPUT_PATH_BEN)

print("Base data loaded.")
print("Shape:", df_base_v2.shape)
print("Columns:", df_base_v2.shape[1])

df_base_v2.head(3)

Input path: /content/drive/MyDrive/divar_project/cleaned_step1.parquet
Exists: True
Base data loaded.
Shape: (999943, 55)
Columns: 55


,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,rent_mode,rent_value,rent_type,price_mode,price_value,credit_mode,credit_value,rent_credit_transform,transformable_price,transformable_credit,transformed_credit,transformable_rent,transformed_rent,land_size,building_size,deed_type,has_business_deed,floor,rooms_count,total_floors_count,unit_per_floor,has_balcony,has_elevator,has_warehouse,has_parking,construction_year,is_rebuilt,has_warm_water_provider,has_heating_system,has_cooling_system,has_restroom,has_security_guard,has_barbecue,building_direction,has_pool,has_jacuzzi,has_sauna,floor_material,property_type,regular_person_capacity,location_latitude,location_longitude,location_radius,created_at_shamsi,created_at_shamsi_readable,construction_year_was_missing,construction_year_imputed
0,temporary-rent,villa,karaj,mehrshahr,2024-08-01,مشاور املاک,۵۰۰متر\n۲۰۰متر بنا دوبلکس\n۳خواب\nاستخر آبگرم ...,باغ ویلا اجاره روزانه استخر داخل لشکرآباد سهیلیه,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,500.0,NaN,<NA>,NaN,3.0,NaN,NaN,False,False,False,False,NaN,False,NaN,NaN,NaN,NaN,False,False,NaN,False,False,False,NaN,NaN,4.0,35.811684,50.936600,500.0,1403-05,مرداد 1403,True,1395.0
1,residential-sell,apartment-sell,tehran,gholhak,2024-05-01,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,NaN,NaN,مقطوع,8.500000e+09,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,60.0,NaN,<NA>,3.0,1.0,NaN,NaN,False,True,True,True,1384.0,False,NaN,NaN,NaN,NaN,False,False,NaN,False,False,False,NaN,NaN,NaN,NaN,NaN,500.0,1403-02,اردیبهشت 1403,False,1384.0
2,residential-rent,apartment-rent,tehran,tohid,2024-10-01,NaN,تخلیه پایان ماه,آپارتمان ۳ خوابه ۱۳۲ متر,مقطوع,26000000.0,NaN,NaN,NaN,مقطوع,750000000.0,False,False,750000000.0,NaN,26000000.0,NaN,NaN,132.0,NaN,<NA>,3.0,3.0,NaN,NaN,False,True,True,True,1401.0,False,NaN,NaN,NaN,NaN,False,False,NaN,False,False,False,NaN,NaN,NaN,35.703865,51.373459,NaN,1403-07,مهر 1403,False,1401.0


<div dir="rtl" align="right">

### 4.1 بررسی ستون‌های لازم برای preprocessing بنیامین

در این مرحله ستون‌هایی که برای ساخت ویژگی‌های قیمت، متراژ، قیمت هدف و flagهای جغرافیایی لازم هستند بررسی می‌شوند.

اگر یکی از این ستون‌ها وجود نداشته باشد، اجرای بخش بنیامین متوقف می‌شود تا ورودی اشتباه وارد مراحل بعدی نشود.

</div>

In [8]:
required_cols = [
    "cat2_slug",
    "cat3_slug",
    "price_value",
    "rent_value",
    "credit_value",
    "building_size",
    "land_size",
    "location_latitude",
    "location_longitude"
]

missing_required = [col for col in required_cols if col not in df_base_v2.columns]

print("Missing required columns:", missing_required)

if missing_required:
    raise ValueError("Some required columns are missing. Stop here and check the input data.")

Missing required columns: []


In [9]:
df_pre_v2 = df_base_v2.copy(deep=False)

print("Base shape:", df_base_v2.shape)
print("Preprocessing v2 shape:", df_pre_v2.shape)

Base shape: (999943, 55)
Preprocessing v2 shape: (999943, 55)


<div dir="rtl" align="right">

### 4.2 اجرای کامل preprocessing بنیامین

در این مرحله کد اصلی بنیامین برای ساخت ستون‌های قیمت، قیمت واحد، قیمت هدف، flagهای اعتبارسنجی و flagهای جغرافیایی اجرا می‌شود.

این کد از روی نوت‌بوک `prep_ben.ipynb` منتقل شده و روی خروجی به‌روزشده‌ی مرحله علی اجرا می‌شود.

در این مرحله هیچ ردیفی حذف نمی‌شود.

</div>

In [10]:
# -----------------------------
# 1. Transaction type
# -----------------------------

cat2 = df_pre_v2["cat2_slug"].astype("string")

df_pre_v2["transaction_type"] = np.select(
    [
        cat2.eq("temporary-rent"),
        cat2.str.endswith("-sell"),
        cat2.str.endswith("-rent"),
    ],
    [
        "temporary_rent",
        "sell",
        "rent",
    ],
    default="other"
)

is_sell = df_pre_v2["transaction_type"].eq("sell")
is_rent = df_pre_v2["transaction_type"].eq("rent")


# -----------------------------
# 2. Positive versions of numeric columns
# -----------------------------

def positive_or_nan(series):
    return series.where(series > 0, np.nan)

df_pre_v2["price_value_pos"] = positive_or_nan(df_pre_v2["price_value"])
df_pre_v2["rent_value_pos"] = positive_or_nan(df_pre_v2["rent_value"])
df_pre_v2["credit_value_pos"] = positive_or_nan(df_pre_v2["credit_value"])
df_pre_v2["building_size_pos"] = positive_or_nan(df_pre_v2["building_size"])
df_pre_v2["land_size_pos"] = positive_or_nan(df_pre_v2["land_size"])


# -----------------------------
# 3. Area used for unit price
# -----------------------------

df_pre_v2["area_for_unit_price"] = df_pre_v2["building_size_pos"]

is_plot_old = df_pre_v2["cat3_slug"].astype("string").eq("plot-old")

df_pre_v2.loc[
    is_plot_old & df_pre_v2["land_size_pos"].notna(),
    "area_for_unit_price"
] = df_pre_v2.loc[
    is_plot_old & df_pre_v2["land_size_pos"].notna(),
    "land_size_pos"
]


# -----------------------------
# 4. Rent-credit equivalent values
# -----------------------------

CREDIT_TO_MONTHLY_RENT_RATE = 30_000 / 1_000_000

has_rent_or_credit = (
    df_pre_v2["rent_value_pos"].notna() |
    df_pre_v2["credit_value_pos"].notna()
)

df_pre_v2["monthly_rent_equivalent"] = np.where(
    is_rent & has_rent_or_credit,
    df_pre_v2["rent_value_pos"].fillna(0) +
    df_pre_v2["credit_value_pos"].fillna(0) * CREDIT_TO_MONTHLY_RENT_RATE,
    np.nan
)

df_pre_v2["credit_equivalent"] = np.where(
    is_rent & has_rent_or_credit,
    df_pre_v2["credit_value_pos"].fillna(0) +
    df_pre_v2["rent_value_pos"].fillna(0) / CREDIT_TO_MONTHLY_RENT_RATE,
    np.nan
)


# -----------------------------
# 5. Unit price and target price
# -----------------------------

valid_area = (
    df_pre_v2["area_for_unit_price"].notna() &
    (df_pre_v2["area_for_unit_price"] > 0)
)

df_pre_v2["sale_price_per_m2"] = np.where(
    is_sell & df_pre_v2["price_value_pos"].notna() & valid_area,
    df_pre_v2["price_value_pos"] / df_pre_v2["area_for_unit_price"],
    np.nan
)

df_pre_v2["monthly_rent_equivalent_per_m2"] = np.where(
    is_rent & df_pre_v2["monthly_rent_equivalent"].notna() & valid_area,
    df_pre_v2["monthly_rent_equivalent"] / df_pre_v2["area_for_unit_price"],
    np.nan
)

df_pre_v2["credit_equivalent_per_m2"] = np.where(
    is_rent & df_pre_v2["credit_equivalent"].notna() & valid_area,
    df_pre_v2["credit_equivalent"] / df_pre_v2["area_for_unit_price"],
    np.nan
)

df_pre_v2["target_price"] = np.select(
    [
        is_sell,
        is_rent,
    ],
    [
        df_pre_v2["price_value_pos"],
        df_pre_v2["monthly_rent_equivalent"],
    ],
    default=np.nan
)

df_pre_v2["target_price_type"] = np.select(
    [
        is_sell,
        is_rent,
    ],
    [
        "sale_total_price",
        "monthly_rent_equivalent",
    ],
    default="not_defined"
)

df_pre_v2["target_price_per_m2"] = np.where(
    df_pre_v2["target_price"].notna() & valid_area,
    df_pre_v2["target_price"] / df_pre_v2["area_for_unit_price"],
    np.nan
)


# -----------------------------
# 6. Basic validation flags
# -----------------------------

df_pre_v2["is_valid_area_basic"] = (
    df_pre_v2["area_for_unit_price"].notna() &
    df_pre_v2["area_for_unit_price"].between(10, 20_000)
)

df_pre_v2["is_valid_sale_price_basic"] = (
    df_pre_v2["price_value"].isna() |
    df_pre_v2["price_value"].between(10_000_000, 500_000_000_000)
)

df_pre_v2["is_valid_rent_value_basic"] = (
    df_pre_v2["rent_value"].isna() |
    df_pre_v2["rent_value"].between(0, 1_000_000_000)
)

df_pre_v2["is_valid_credit_value_basic"] = (
    df_pre_v2["credit_value"].isna() |
    df_pre_v2["credit_value"].between(0, 50_000_000_000)
)

df_pre_v2["is_valid_target_price_basic"] = (
    df_pre_v2["target_price"].isna() |
    (
        is_sell &
        df_pre_v2["target_price"].between(10_000_000, 500_000_000_000)
    ) |
    (
        is_rent &
        df_pre_v2["target_price"].between(0, 1_500_000_000)
    )
)


# -----------------------------
# 7. Quantile-based validation flags by cat3_slug
# -----------------------------

def add_group_quantile_flag(
    data,
    value_col,
    group_col,
    flag_col,
    lower_q=0.01,
    upper_q=0.99,
    min_group_count=100
):
    valid = data[value_col].notna()

    group_counts = (
        data.loc[valid]
        .groupby(group_col, observed=True)[value_col]
        .transform("count")
    )

    lower_bounds = (
        data.loc[valid]
        .groupby(group_col, observed=True)[value_col]
        .transform(lambda s: s.quantile(lower_q))
    )

    upper_bounds = (
        data.loc[valid]
        .groupby(group_col, observed=True)[value_col]
        .transform(lambda s: s.quantile(upper_q))
    )

    result = pd.Series(pd.NA, index=data.index, dtype="boolean")
    valid_index = data.loc[valid].index

    result.loc[valid_index] = (
        (group_counts >= min_group_count) &
        (data.loc[valid, value_col] >= lower_bounds) &
        (data.loc[valid, value_col] <= upper_bounds)
    ).astype("boolean")

    data[flag_col] = result
    return data


df_pre_v2 = add_group_quantile_flag(
    data=df_pre_v2,
    value_col="area_for_unit_price",
    group_col="cat3_slug",
    flag_col="is_valid_area_q01_q99_by_cat3",
    lower_q=0.01,
    upper_q=0.99,
    min_group_count=100
)

df_pre_v2 = add_group_quantile_flag(
    data=df_pre_v2,
    value_col="target_price_per_m2",
    group_col="cat3_slug",
    flag_col="is_valid_target_price_per_m2_q01_q99_by_cat3",
    lower_q=0.01,
    upper_q=0.99,
    min_group_count=100
)


# -----------------------------
# 8. Final price-analysis validation flag
# -----------------------------

df_pre_v2["is_valid_for_price_analysis"] = (
    df_pre_v2["transaction_type"].isin(["sell", "rent"]) &
    df_pre_v2["is_valid_area_basic"] &
    df_pre_v2["is_valid_target_price_basic"] &
    df_pre_v2["is_valid_area_q01_q99_by_cat3"].fillna(False) &
    df_pre_v2["is_valid_target_price_per_m2_q01_q99_by_cat3"].fillna(False)
)


# -----------------------------
# 9. Geo flags
# -----------------------------

has_geo = (
    df_pre_v2["location_latitude"].notna() &
    df_pre_v2["location_longitude"].notna()
)

df_pre_v2["has_geo"] = has_geo

df_pre_v2["is_valid_geo_basic"] = pd.Series(
    pd.NA,
    index=df_pre_v2.index,
    dtype="boolean"
)

df_pre_v2.loc[has_geo, "is_valid_geo_basic"] = (
    df_pre_v2.loc[has_geo, "location_latitude"].between(24, 40) &
    df_pre_v2.loc[has_geo, "location_longitude"].between(43, 64)
)


print("Benyamin preprocessing v2 completed.")
print("Shape:", df_pre_v2.shape)
print("Columns:", df_pre_v2.shape[1])

Benyamin preprocessing v2 completed.
Shape: (999943, 80)
Columns: 80


<div dir="rtl" align="right">

### 4.3 گزارش خلاصه خروجی بنیامین

در این مرحله خلاصه‌ای از خروجی preprocessing بنیامین ساخته می‌شود.

این گزارش تعداد کل رکوردها، تعداد آگهی‌های فروش، اجاره، اجاره موقت، رکوردهای دارای قیمت هدف، رکوردهای معتبر برای تحلیل قیمت و وضعیت اولیه مختصات جغرافیایی را نشان می‌دهد.

</div>

In [11]:
preprocessing_report_v2 = pd.DataFrame({
    "item": [
        "total_rows",
        "sell_ads",
        "rent_ads",
        "temporary_rent_ads",
        "other_ads",
        "rows_with_target_price",
        "valid_for_price_analysis",
        "invalid_for_price_analysis",
        "rows_with_geo",
        "invalid_geo_rows",
        "invalid_area_basic",
        "invalid_target_price_basic"
    ],
    "count": [
        len(df_pre_v2),
        (df_pre_v2["transaction_type"] == "sell").sum(),
        (df_pre_v2["transaction_type"] == "rent").sum(),
        (df_pre_v2["transaction_type"] == "temporary_rent").sum(),
        (df_pre_v2["transaction_type"] == "other").sum(),
        df_pre_v2["target_price"].notna().sum(),
        df_pre_v2["is_valid_for_price_analysis"].sum(),
        (~df_pre_v2["is_valid_for_price_analysis"]).sum(),
        df_pre_v2["has_geo"].sum(),
        (df_pre_v2["is_valid_geo_basic"] == False).sum(),
        (~df_pre_v2["is_valid_area_basic"]).sum(),
        (~df_pre_v2["is_valid_target_price_basic"]).sum()
    ]
})

preprocessing_report_v2["percent"] = (
    preprocessing_report_v2["count"] / len(df_pre_v2) * 100
).round(2)

display(preprocessing_report_v2)

,item,count,percent
0,total_rows,999943,100.00
1,sell_ads,597552,59.76
2,rent_ads,353092,35.31
3,temporary_rent_ads,29897,2.99
4,other_ads,19402,1.94
5,rows_with_target_price,917334,91.74
6,valid_for_price_analysis,877587,87.76
7,invalid_for_price_analysis,122356,12.24
8,rows_with_geo,655594,65.56
9,invalid_geo_rows,18,0.00


<div dir="rtl" align="right">

### 4.4 ذخیره خروجی مرحله بنیامین

در این مرحله خروجی نهایی preprocessing بنیامین با نام `cleaned_step2_benyamin_v2.parquet` ذخیره می‌شود.

این فایل ورودی مرحله بعدی، یعنی preprocessing جغرافیایی لیلا، خواهد بود.

</div>

In [12]:
OUTPUT_PATH_V2 = "/content/drive/MyDrive/divar_project/cleaned_step2_benyamin_v2.parquet"

output_dir = os.path.dirname(OUTPUT_PATH_V2)

if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

df_pre_v2.to_parquet(OUTPUT_PATH_V2, index=False)

print("Saved to:", OUTPUT_PATH_V2)
print("Shape:", df_pre_v2.shape)
print("Columns:", df_pre_v2.shape[1])

Saved to: /content/drive/MyDrive/divar_project/cleaned_step2_benyamin_v2.parquet
Shape: (999943, 80)
Columns: 80


<div dir="rtl" align="right">

### 4.5 بررسی فایل ذخیره‌شده بنیامین

در این مرحله فایل خروجی بنیامین دوباره خوانده می‌شود تا مطمئن شویم ذخیره‌سازی درست انجام شده و تعداد ردیف‌ها، ستون‌ها و شاخص‌های اصلی با خروجی مورد انتظار هماهنگ هستند.

</div>

In [13]:
# فایل ذخیره‌شده بنیامین را دوباره می‌خوانیم
test_ben = pd.read_parquet(OUTPUT_PATH_V2)

print("Shape:", test_ben.shape)
print("Columns:", test_ben.shape[1])

print("\nMain checks:")
print("rows_with_target_price:", test_ben["target_price"].notna().sum())
print("valid_for_price_analysis:", test_ben["is_valid_for_price_analysis"].sum())
print("invalid_for_price_analysis:", (~test_ben["is_valid_for_price_analysis"]).sum())
print("rows_with_geo:", test_ben["has_geo"].sum())
print("invalid_geo_rows:", (test_ben["is_valid_geo_basic"] == False).sum())

Shape: (999943, 80)
Columns: 80

Main checks:
rows_with_target_price: 917334
valid_for_price_analysis: 877587
invalid_for_price_analysis: 122356
rows_with_geo: 655594
invalid_geo_rows: 18


<div dir="rtl" align="right">

## 5. preprocessing جغرافیایی علی

در این بخش، خروجی مرحله بنیامین خوانده می‌شود و پردازش‌های جغرافیایی روی آن انجام می‌شود.

ورودی این مرحله فایل `cleaned_step2_benyamin_v2.parquet` است.

در این مرحله ستون‌های جغرافیایی بررسی می‌شوند، مختصات معتبر مشخص می‌شوند و برای ردیف‌های معتبر، تبدیل مختصات latitude/longitude به UTM انجام می‌شود.

</div>

In [14]:
data_path = "/content/drive/MyDrive/divar_project/cleaned_step2_benyamin_v2.parquet"

df = pd.read_parquet(data_path)

print("Shape:", df.shape)
print("Number of columns:", len(df.columns))

df.head()

Shape: (999943, 80)
Number of columns: 80


,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,rent_mode,rent_value,rent_type,price_mode,price_value,credit_mode,credit_value,rent_credit_transform,transformable_price,transformable_credit,transformed_credit,transformable_rent,transformed_rent,land_size,building_size,deed_type,has_business_deed,floor,rooms_count,total_floors_count,unit_per_floor,has_balcony,has_elevator,has_warehouse,has_parking,construction_year,is_rebuilt,has_warm_water_provider,has_heating_system,has_cooling_system,has_restroom,has_security_guard,has_barbecue,building_direction,has_pool,has_jacuzzi,has_sauna,floor_material,property_type,regular_person_capacity,location_latitude,location_longitude,location_radius,created_at_shamsi,created_at_shamsi_readable,construction_year_was_missing,construction_year_imputed,transaction_type,price_value_pos,rent_value_pos,credit_value_pos,building_size_pos,land_size_pos,area_for_unit_price,monthly_rent_equivalent,credit_equivalent,sale_price_per_m2,monthly_rent_equivalent_per_m2,credit_equivalent_per_m2,target_price,target_price_type,target_price_per_m2,is_valid_area_basic,is_valid_sale_price_basic,is_valid_rent_value_basic,is_valid_credit_value_basic,is_valid_target_price_basic,is_valid_area_q01_q99_by_cat3,is_valid_target_price_per_m2_q01_q99_by_cat3,is_valid_for_price_analysis,has_geo,is_valid_geo_basic
0,temporary-rent,villa,karaj,mehrshahr,2024-08-01,مشاور املاک,۵۰۰متر\n۲۰۰متر بنا دوبلکس\n۳خواب\nاستخر آبگرم ...,باغ ویلا اجاره روزانه استخر داخل لشکرآباد سهیلیه,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,500.0,NaN,<NA>,NaN,3.0,NaN,NaN,False,False,False,False,NaN,False,NaN,NaN,NaN,NaN,False,False,NaN,False,False,False,NaN,NaN,4.0,35.811684,50.936600,500.0,1403-05,مرداد 1403,True,1395.0,temporary_rent,NaN,NaN,NaN,500.0,NaN,500.0,NaN,NaN,NaN,NaN,NaN,NaN,not_defined,NaN,True,True,True,True,True,True,<NA>,False,True,True
1,residential-sell,apartment-sell,tehran,gholhak,2024-05-01,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,NaN,NaN,مقطوع,8.500000e+09,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,60.0,NaN,<NA>,3.0,1.0,NaN,NaN,False,True,True,True,1384.0,False,NaN,NaN,NaN,NaN,False,False,NaN,False,False,False,NaN,NaN,NaN,NaN,NaN,500.0,1403-02,اردیبهشت 1403,False,1384.0,sell,8.500000e+09,NaN,NaN,60.0,NaN,60.0,NaN,NaN,1.416667e+08,NaN,NaN,8.500000e+09,sale_total_price,1.416667e+08,True,True,True,True,True,True,True,True,False,<NA>
2,residential-rent,apartment-rent,tehran,tohid,2024-10-01,NaN,تخلیه پایان ماه,آپارتمان ۳ خوابه ۱۳۲ متر,مقطوع,26000000.0,NaN,NaN,NaN,مقطوع,750000000.0,False,False,750000000.0,NaN,26000000.0,NaN,NaN,132.0,NaN,<NA>,3.0,3.0,NaN,NaN,False,True,True,True,1401.0,False,NaN,NaN,NaN,NaN,False,False,NaN,False,False,False,NaN,NaN,NaN,35.703865,51.373459,NaN,1403-07,مهر 1403,False,1401.0,rent,NaN,26000000.0,750000000.0,132.0,NaN,132.0,48500000.0,1.616667e+09,NaN,3.674242e+05,1.224747e+07,4.850000e+07,monthly_rent_equivalent,3.674242e+05,True,True,True,True,True,True,True,True,True,True
3,commercial-rent,office-rent,tehran,elahiyeh,2024-06-01,NaN,فرشته تاپ لوکیشن\n۹۰ متر موقعیت اداری\nیک اتاق...,فرشته ۹۰ متر دفتر کار مدرن موقعیت اداری,مقطوع,95000000.0,NaN,NaN,NaN,مقطوع,950000000.0,False,False,950000000.0,NaN,95000000.0,NaN,NaN,90.0,NaN,<NA>,4.0,1.0,NaN,NaN,False,True,False,True,1400.0,False,NaN,NaN,NaN,NaN,False,False,NaN,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,1403-03,خرداد 1403,False,1400.0,rent,NaN,95000000.0,950000000.0,90.0,NaN,90.0,123500000.0,4.116667e+09,NaN,1.372222e+06,4.574074e+07,1.235000e+08,monthly_rent_equivalent,1.372222e+06,True,True,True,True,True,True,True,True,False,<NA>
4,residential-sell,apartment-sell,mashhad,emamreza,2024-05-01,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,NaN,NaN,NaN,مقطوع,5.750000e+09,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,115.0,single_page,<NA>,4.0,2.0,6.0,NaN,True,True,True,True,1403.0,False,package,shoofaj,air_conditioner,s

<div dir="rtl" align="right">

### 5.1 بررسی ستون‌های موجود

در این مرحله نام همه ستون‌های دیتاست بررسی می‌شود تا مطمئن شویم ستون‌های لازم برای پردازش جغرافیایی و تبدیل UTM در فایل وجود دارند.

</div>

In [15]:
for i, col in enumerate(df.columns, start=1):
    print(i, col)

1 cat2_slug
2 cat3_slug
3 city_slug
4 neighborhood_slug
5 created_at_month
6 user_type
7 description
8 title
9 rent_mode
10 rent_value
11 rent_type
12 price_mode
13 price_value
14 credit_mode
15 credit_value
16 rent_credit_transform
17 transformable_price
18 transformable_credit
19 transformed_credit
20 transformable_rent
21 transformed_rent
22 land_size
23 building_size
24 deed_type
25 has_business_deed
26 floor
27 rooms_count
28 total_floors_count
29 unit_per_floor
30 has_balcony
31 has_elevator
32 has_warehouse
33 has_parking
34 construction_year
35 is_rebuilt
36 has_warm_water_provider
37 has_heating_system
38 has_cooling_system
39 has_restroom
40 has_security_guard
41 has_barbecue
42 building_direction
43 has_pool
44 has_jacuzzi
45 has_sauna
46 floor_material
47 property_type
48 regular_person_capacity
49 location_latitude
50 location_longitude
51 location_radius
52 created_at_shamsi
53 created_at_shamsi_readable
54 construction_year_was_missing
55 construction_year_imputed
56 tra

<div dir="rtl" align="right">

### 5.2 بررسی خلاصه ستون‌های جغرافیایی

در این مرحله ستون‌های جغرافیایی اصلی بررسی می‌شوند.

برای هر ستون، نوع داده، تعداد مقدارهای گم‌شده، درصد مقدارهای گم‌شده و تعداد مقدارهای یکتا محاسبه می‌شود.

</div>

In [16]:
geo_cols = [
    "city_slug",
    "neighborhood_slug",
    "location_latitude",
    "location_longitude",
    "location_radius",
    "has_geo",
    "is_valid_geo_basic",
]

geo_summary = pd.DataFrame({
    "dtype": df[geo_cols].dtypes,
    "missing_count": df[geo_cols].isna().sum(),
    "missing_percent": df[geo_cols].isna().mean() * 100,
    "n_unique": df[geo_cols].nunique(dropna=True),
})

geo_summary

,dtype,missing_count,missing_percent,n_unique
city_slug,category,0,0.000000,421
neighborhood_slug,category,562854,56.288608,1188
location_latitude,float64,344349,34.436863,387482
location_longitude,float64,344349,34.436863,429426
location_radius,float64,660249,66.028664,3
has_geo,bool,0,0.000000,2
is_valid_geo_basic,boolean,344349,34.436863,2


<div dir="rtl" align="right">

### 5.3 بررسی flagهای جغرافیایی

در این مرحله بررسی می‌شود که ستون `has_geo` با وجود یا نبودن latitude و longitude هماهنگ است یا نه.

همچنین تعداد ردیف‌هایی که مختصات دارند اما در اعتبارسنجی اولیه جغرافیایی نامعتبر شناخته شده‌اند بررسی می‌شود.

</div>

In [17]:
lat_col = "location_latitude"
lon_col = "location_longitude"

calculated_has_geo = df[lat_col].notna() & df[lon_col].notna()

print("has_geo value counts:")
print(df["has_geo"].value_counts(dropna=False))

print("\nis_valid_geo_basic value counts:")
print(df["is_valid_geo_basic"].value_counts(dropna=False))

print("\nMismatch between calculated_has_geo and has_geo:")
print((calculated_has_geo != df["has_geo"]).sum())

print("\nRows with geo but invalid basic geo:")
print((df["has_geo"] & (df["is_valid_geo_basic"] == False)).sum())

has_geo value counts:
has_geo
True     655594
False    344349
Name: count, dtype: int64

is_valid_geo_basic value counts:
is_valid_geo_basic
True     655576
<NA>     344349
False        18
Name: count, dtype: Int64

Mismatch between calculated_has_geo and has_geo:
0

Rows with geo but invalid basic geo:
18


<div dir="rtl" align="right">

### 5.4 بررسی مختصات جغرافیایی نامعتبر

در این مرحله ردیف‌هایی بررسی می‌شوند که latitude و longitude دارند، اما در اعتبارسنجی اولیه جغرافیایی نامعتبر تشخیص داده شده‌اند.

این بررسی فقط برای مشاهده و کنترل کیفیت داده است و در این مرحله ردیفی حذف نمی‌شود.

</div>

In [18]:
invalid_geo = df[df["has_geo"] & (df["is_valid_geo_basic"] == False)].copy()

invalid_geo[
    [
        "city_slug",
        "neighborhood_slug",
        "cat2_slug",
        "cat3_slug",
        "location_latitude",
        "location_longitude",
        "location_radius",
    ]
]

,city_slug,neighborhood_slug,cat2_slug,cat3_slug,location_latitude,location_longitude,location_radius
10895,nahavand,NaN,residential-sell,apartment-sell,23.636976,52.240677,NaN
71647,qeshm,NaN,temporary-rent,villa,23.636976,55.139008,NaN
145957,mashhad,torbatheydarieh,residential-rent,house-villa-rent,32.822731,74.511620,NaN
420979,fasa-city,NaN,residential-sell,apartment-sell,23.636976,52.334061,NaN
460012,konarak,NaN,residential-sell,plot-old,23.636976,59.933167,NaN
494275,bandar-abbas,NaN,residential-sell,apartment-sell,23.636976,55.105362,NaN
510854,tabriz,NaN,real-estate-services,partnership,36.005264,40.162369,NaN
565173,qeshm,NaN,temporary-rent,suite-apartment,23.636976,54.584198,NaN
565402,hashtgerd-city,NaN,residential-sell,plot-old,36.983528,40.162369,NaN
672741,bandar-abbas,NaN,residential-sell,apartment-sell,23.777609,55.415295,NaN


<div dir="rtl" align="right">

### 5.5 بررسی ستون location_radius

در این مرحله مقدارهای ستون `location_radius` بررسی می‌شود.

این ستون دقت تقریبی موقعیت مکانی آگهی را نشان می‌دهد، اما برای همه ردیف‌ها مقدار ندارد.  
در این مرحله فقط توزیع مقدارهای آن بررسی می‌شود و تغییری در دیتاست انجام نمی‌شود.

</div>

In [19]:
df["location_radius"].value_counts(dropna=False)

,count
location_radius,
NaN,660249
500.0,314790
0.0,22860
300.0,2044


<div dir="rtl" align="right">

### 5.6 بررسی location_radius برای مختصات معتبر

در این مرحله مقدارهای `location_radius` فقط برای ردیف‌هایی بررسی می‌شود که مختصات جغرافیایی معتبر دارند.

این بررسی کمک می‌کند ببینیم در داده‌های قابل استفاده‌ی جغرافیایی، دقت مکانی آگهی‌ها چه وضعیتی دارد.

</div>

In [20]:
valid_geo = df[df["is_valid_geo_basic"] == True].copy()

valid_geo["location_radius"].value_counts(dropna=False)

,count
location_radius,
NaN,414376
500.0,223904
0.0,15302
300.0,1994


<div dir="rtl" align="right">

### 5.7 بررسی بازه مختصات معتبر

در این مرحله فقط ردیف‌هایی بررسی می‌شوند که مختصات جغرافیایی معتبر دارند.

با استفاده از آمار توصیفی، بازه‌ی latitude و longitude بررسی می‌شود تا مطمئن شویم مختصات معتبر در محدوده‌ی منطقی ایران قرار دارند.

</div>

In [21]:
valid_geo[["location_latitude", "location_longitude"]].describe(
    percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
)

,location_latitude,location_longitude
count,655576.000000,655576.000000
mean,34.982323,51.629625
std,2.378646,3.160582
min,24.154272,43.135620
0.1%,25.322581,44.822493
1%,27.191817,45.067278
5%,29.608631,46.422072
50%,35.723312,51.345779
95%,37.561452,59.500443
99%,38.249336,59.776517


<div dir="rtl" align="right">

### 5.8 بررسی پوشش جغرافیایی بر اساس cat2_slug

در این مرحله بررسی می‌شود که در هر دسته اصلی آگهی، چه تعداد ردیف مختصات جغرافیایی معتبر دارند.

این بررسی نشان می‌دهد پوشش داده‌های جغرافیایی در دسته‌های مختلف تقریباً چقدر است.

</div>

In [22]:
geo_coverage_cat2 = (
    df
    .groupby("cat2_slug", observed=True)
    .agg(
        total_rows=("cat2_slug", "size"),
        valid_geo_rows=("is_valid_geo_basic", lambda s: (s == True).sum()),
    )
    .reset_index()
)

geo_coverage_cat2["valid_geo_percent"] = (
    geo_coverage_cat2["valid_geo_rows"]
    / geo_coverage_cat2["total_rows"]
    * 100
).round(2)

geo_coverage_cat2.sort_values("total_rows", ascending=False)

,cat2_slug,total_rows,valid_geo_rows,valid_geo_percent
4,residential-sell,558692,364505,65.24
3,residential-rent,276528,184142,66.59
0,commercial-rent,76564,50937,66.53
1,commercial-sell,38860,25532,65.70
5,temporary-rent,29897,19458,65.08
2,real-estate-services,19402,11002,56.71


<div dir="rtl" align="right">

### 5.9 بررسی پوشش جغرافیایی بر اساس cat3_slug

در این مرحله پوشش مختصات جغرافیایی معتبر در دسته‌بندی‌های سطح سوم بررسی می‌شود.

برای خواناتر بودن خروجی، فقط دسته‌هایی نمایش داده می‌شوند که بیشترین تعداد آگهی را دارند.

</div>

In [23]:
geo_by_cat3 = (
    df.groupby("cat3_slug", observed=True)
      .agg(
          total_rows=("cat3_slug", "size"),
          has_geo_count=("has_geo", "sum"),
          valid_geo_count=("is_valid_geo_basic", lambda s: (s == True).sum())
      )
      .reset_index()
)

geo_by_cat3["has_geo_percent"] = geo_by_cat3["has_geo_count"] / geo_by_cat3["total_rows"] * 100
geo_by_cat3["valid_geo_percent"] = geo_by_cat3["valid_geo_count"] / geo_by_cat3["total_rows"] * 100

geo_by_cat3.sort_values("total_rows", ascending=False).head(30)

,cat3_slug,total_rows,has_geo_count,valid_geo_count,has_geo_percent,valid_geo_percent
1,apartment-sell,303372,205068,205062,67.596219,67.594241
0,apartment-rent,211853,145268,145266,68.570188,68.569244
9,plot-old,133570,84745,84742,63.446133,63.443887
3,house-villa-sell,121750,74701,74701,61.356057,61.356057
2,house-villa-rent,64675,38879,38876,60.114418,60.109780
11,shop-rent,45993,30313,30313,65.907856,65.907856
12,shop-sell,21855,14259,14259,65.243651,65.243651
6,office-rent,21416,14494,14494,67.678371,67.678371
13,suite-apartment,16460,10506,10505,63.827461,63.821385
10,presell,15780,8989,8989,56.964512,56.964512


<div dir="rtl" align="right">

### 5.10 ساخت flag نهایی برای تحلیل جغرافیایی

در این مرحله ستون `is_valid_geo_for_analysis` ساخته می‌شود.

این ستون مشخص می‌کند کدام ردیف‌ها مختصات جغرافیایی معتبر دارند و می‌توانند در تحلیل‌های مکانی، نقشه و clustering استفاده شوند.

</div>

In [24]:
df["is_valid_geo_for_analysis"] = (
    df["is_valid_geo_basic"] == True
).fillna(False).astype(bool)

print(df["is_valid_geo_for_analysis"].value_counts(dropna=False))

print("\nValid geo rows:")
print(df["is_valid_geo_for_analysis"].sum())

print("\nInvalid or missing geo rows:")
print((~df["is_valid_geo_for_analysis"]).sum())

print("\nColumn dtype:")
print(df["is_valid_geo_for_analysis"].dtype)

is_valid_geo_for_analysis
True     655576
False    344367
Name: count, dtype: int64

Valid geo rows:
655576

Invalid or missing geo rows:
344367

Column dtype:
bool


<div dir="rtl" align="right">

### 5.11 نصب کتابخانه UTM

در این مرحله کتابخانه `utm` نصب می‌شود تا بتوانیم مختصات latitude و longitude را به مختصات UTM تبدیل کنیم.

</div>

In [25]:
!pip install utm

In [26]:
import utm

sample_lat = 35.6892
sample_lon = 51.3890

utm.from_latlon(sample_lat, sample_lon)

(np.float64(535196.781829007), np.float64(3949546.7888794737), 39, 'S')

<div dir="rtl" align="right">

### 5.12 تعریف تابع تبدیل مختصات به UTM

در این مرحله تابعی تعریف می‌شود که مقدارهای `latitude` و `longitude` را دریافت می‌کند و آن‌ها را به مختصات UTM تبدیل می‌کند.

خروجی تابع شامل `utm_easting`، `utm_northing`، `utm_zone_number` و `utm_zone_letter` است.

اگر مختصات خالی یا نامعتبر باشد، خروجی تابع مقدار خالی برمی‌گرداند.

</div>

In [27]:
def latlon_to_utm(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return pd.Series({
            "utm_easting": np.nan,
            "utm_northing": np.nan,
            "utm_zone_number": pd.NA,
            "utm_zone_letter": pd.NA,
        })

    try:
        easting, northing, zone_number, zone_letter = utm.from_latlon(lat, lon)

        return pd.Series({
            "utm_easting": easting,
            "utm_northing": northing,
            "utm_zone_number": zone_number,
            "utm_zone_letter": zone_letter,
        })

    except Exception:
        return pd.Series({
            "utm_easting": np.nan,
            "utm_northing": np.nan,
            "utm_zone_number": pd.NA,
            "utm_zone_letter": pd.NA,
        })

In [28]:
latlon_to_utm(35.6892, 51.3890)

,0
utm_easting,535196.781829
utm_northing,3949546.788879
utm_zone_number,39
utm_zone_letter,S


<div dir="rtl" align="right">

### 5.13 تست تبدیل UTM روی چند ردیف واقعی

در این مرحله تابع تبدیل UTM روی چند ردیف دارای مختصات معتبر تست می‌شود.

این تست قبل از اجرای تبدیل روی کل دیتاست انجام می‌شود تا مطمئن شویم خروجی تابع برای داده‌های واقعی درست و قابل استفاده است.

</div>

In [29]:
sample_geo = df[df["is_valid_geo_for_analysis"]].head(5).copy()

sample_utm = sample_geo.apply(
    lambda row: latlon_to_utm(
        row["location_latitude"],
        row["location_longitude"]
    ),
    axis=1
)

pd.concat(
    [
        sample_geo[
            [
                "city_slug",
                "neighborhood_slug",
                "location_latitude",
                "location_longitude",
            ]
        ],
        sample_utm,
    ],
    axis=1
)

,city_slug,neighborhood_slug,location_latitude,location_longitude,utm_easting,utm_northing,utm_zone_number,utm_zone_letter
0,karaj,mehrshahr,35.811684,50.936600,494272.329998,3.963064e+06,39,S
2,tehran,tohid,35.703865,51.373459,533784.424602,3.951168e+06,39,S
7,tehran,dardasht,35.729832,51.505466,545711.558185,3.954101e+06,39,S
8,mahdasht-city,NaN,35.712364,50.794781,481437.130969,3.952066e+06,39,S
10,pardis-city,NaN,35.778664,51.757549,568467.028494,3.959664e+06,39,S


<div dir="rtl" align="right">

### 5.14 تبدیل مختصات معتبر به UTM

در این مرحله مختصات latitude و longitude برای همه ردیف‌هایی که مختصات معتبر دارند، به UTM تبدیل می‌شود.

برای ردیف‌هایی که مختصات معتبر ندارند، ستون‌های UTM خالی باقی می‌مانند.

</div>

In [30]:
utm_cols = [
    "utm_easting",
    "utm_northing",
    "utm_zone_number",
    "utm_zone_letter",
]

valid_geo_mask = df["is_valid_geo_for_analysis"]

df[utm_cols] = np.nan

df.loc[valid_geo_mask, utm_cols] = df.loc[valid_geo_mask].apply(
    lambda row: latlon_to_utm(
        row["location_latitude"],
        row["location_longitude"]
    ),
    axis=1
)

df[utm_cols].isna().sum()

/tmp/ipykernel_61213/3057417620.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['S' 'S' 'S' ... 'S' 'S' 'S']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[valid_geo_mask, utm_cols] = df.loc[valid_geo_mask].apply(


,0
utm_easting,344367
utm_northing,344367
utm_zone_number,344367
utm_zone_letter,344367


<div dir="rtl" align="right">

### 5.15 اصلاح نوع داده ستون‌های UTM

در این مرحله نوع داده ستون‌های مربوط به zone اصلاح می‌شود.

ستون `utm_zone_number` به عدد صحیح nullable و ستون `utm_zone_letter` به string تبدیل می‌شود تا warning مرحله قبل برطرف شود و داده‌ها ساختار درست‌تری داشته باشند.

</div>

In [31]:
df["utm_zone_number"] = df["utm_zone_number"].astype("Int64")
df["utm_zone_letter"] = df["utm_zone_letter"].astype("string")

print(df[utm_cols].dtypes)

print("\nMissing values in UTM columns:")
print(df[utm_cols].isna().sum())

utm_easting               float64
utm_northing              float64
utm_zone_number             Int64
utm_zone_letter    string[python]
dtype: object

Missing values in UTM columns:
utm_easting        344367
utm_northing       344367
utm_zone_number    344367
utm_zone_letter    344367
dtype: int64


<div dir="rtl" align="right">

### 5.16 بررسی zoneهای UTM

در این مرحله بررسی می‌شود که مختصات معتبر در چه zoneهای UTM قرار گرفته‌اند.

چون ایران در چند zone مختلف UTM قرار دارد، نگه داشتن `utm_zone_number` و `utm_zone_letter` برای تحلیل‌های مکانی بعدی مهم است.

</div>

In [32]:
utm_zone_counts = (
    df[df["is_valid_geo_for_analysis"]]
    .groupby(["utm_zone_number", "utm_zone_letter"])
    .size()
    .reset_index(name="count")
)

utm_zone_counts["percent"] = (
    utm_zone_counts["count"]
    / df["is_valid_geo_for_analysis"].sum()
    * 100
).round(6)

utm_zone_counts.sort_values(
    ["utm_zone_number", "utm_zone_letter"]
)

,utm_zone_number,utm_zone_letter,count,percent
0,38,R,36,0.005491
1,38,S,54417,8.300639
2,39,R,52891,8.067867
3,39,S,452153,68.970341
4,40,R,25027,3.817559
5,40,S,64641,9.860184
6,41,R,5951,0.907751
7,41,S,460,0.070167


<div dir="rtl" align="right">

### 5.17 بررسی نهایی ستون‌های UTM

در این مرحله بررسی می‌شود که برای همه ردیف‌های دارای مختصات معتبر، ستون‌های UTM ساخته شده باشند.

همچنین بررسی می‌شود ردیف‌هایی که مختصات معتبر ندارند، مقدار UTM نداشته باشند.

</div>

In [33]:
print("Missing values in UTM columns:")
print(df[utm_cols].isna().sum())

print("\nRows with valid geo but missing UTM:")
print(
    df.loc[
        df["is_valid_geo_for_analysis"],
        utm_cols
    ].isna().any(axis=1).sum()
)

print("\nRows without valid geo but having UTM:")
print(
    df.loc[
        ~df["is_valid_geo_for_analysis"],
        utm_cols
    ].notna().any(axis=1).sum()
)

df.loc[
    df["is_valid_geo_for_analysis"],
    [
        "city_slug",
        "location_latitude",
        "location_longitude",
        "utm_easting",
        "utm_northing",
        "utm_zone_number",
        "utm_zone_letter",
    ]
].head()

Missing values in UTM columns:
utm_easting        344367
utm_northing       344367
utm_zone_number    344367
utm_zone_letter    344367
dtype: int64

Rows with valid geo but missing UTM:
0

Rows without valid geo but having UTM:
0


,city_slug,location_latitude,location_longitude,utm_easting,utm_northing,utm_zone_number,utm_zone_letter
0,karaj,35.811684,50.936600,494272.329998,3.963064e+06,39,S
2,tehran,35.703865,51.373459,533784.424602,3.951168e+06,39,S
7,tehran,35.729832,51.505466,545711.558185,3.954101e+06,39,S
8,mahdasht-city,35.712364,50.794781,481437.130969,3.952066e+06,39,S
10,pardis-city,35.778664,51.757549,568467.028494,3.959664e+06,39,S


<div dir="rtl" align="right">

### 5.18 ذخیره خروجی نهایی بخش جغرافیایی

در این مرحله خروجی نهایی preprocessing جغرافیایی ذخیره می‌شود.

این فایل شامل خروجی مرحله بنیامین به‌همراه ستون‌های جدید `is_valid_geo_for_analysis` و ستون‌های UTM است.

خروجی این مرحله با نام `cleaned_step3_geo.parquet` ذخیره می‌شود.

</div>

In [34]:
output_path = "/content/drive/MyDrive/divar_project/cleaned_step3_geo.parquet"

df.to_parquet(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", df.shape)

print("\nNew geo columns:")
print([
    "is_valid_geo_for_analysis",
    "utm_easting",
    "utm_northing",
    "utm_zone_number",
    "utm_zone_letter",
])

Saved to: /content/drive/MyDrive/divar_project/cleaned_step3_geo.parquet
Shape: (999943, 85)

New geo columns:
['is_valid_geo_for_analysis', 'utm_easting', 'utm_northing', 'utm_zone_number', 'utm_zone_letter']


<div dir="rtl" align="right">

### 5.19 بررسی فایل نهایی ذخیره‌شده

در این مرحله فایل `cleaned_step3_geo.parquet` دوباره خوانده می‌شود تا مطمئن شویم ذخیره‌سازی درست انجام شده است.

همچنین نوع داده ستون‌های جدید و تعداد مقدارهای گم‌شده آن‌ها بررسی می‌شود.

</div>

In [35]:
test_geo = pd.read_parquet(output_path)

print("Shape:", test_geo.shape)
print("Columns:", test_geo.shape[1])

new_geo_cols = [
    "is_valid_geo_for_analysis",
    "utm_easting",
    "utm_northing",
    "utm_zone_number",
    "utm_zone_letter",
]

print("\nDtypes:")
print(test_geo[new_geo_cols].dtypes)

print("\nMissing values:")
print(test_geo[new_geo_cols].isna().sum())

print("\nValid geo rows:")
print(test_geo["is_valid_geo_for_analysis"].sum())

Shape: (999943, 85)
Columns: 85

Dtypes:
is_valid_geo_for_analysis              bool
utm_easting                         float64
utm_northing                        float64
utm_zone_number                       Int64
utm_zone_letter              string[python]
dtype: object

Missing values:
is_valid_geo_for_analysis         0
utm_easting                  344367
utm_northing                 344367
utm_zone_number              344367
utm_zone_letter              344367
dtype: int64

Valid geo rows:
655576


<div dir="rtl" align="right">

### 5.20 جمع‌بندی بخش جغرافیایی

در بخش جغرافیایی، فایل خروجی بنیامین یعنی `cleaned_step2_benyamin_v2.parquet` خوانده شد.

ابتدا ستون‌های جغرافیایی بررسی شدند و مشخص شد از مجموع `999,943` ردیف، تعداد `655,594` ردیف دارای مختصات latitude و longitude هستند.

از بین ردیف‌های دارای مختصات، تعداد `655,576` ردیف مختصات معتبر دارند و `18` ردیف مختصات نامعتبر دارند.

سپس ستون `is_valid_geo_for_analysis` ساخته شد تا ردیف‌های قابل استفاده برای تحلیل مکانی مشخص شوند.

در ادامه، برای ردیف‌های دارای مختصات معتبر، مختصات latitude و longitude به UTM تبدیل شد و ستون‌های زیر به دیتاست اضافه شدند:

`utm_easting`

`utm_northing`

`utm_zone_number`

`utm_zone_letter`

در نهایت فایل خروجی با نام `cleaned_step3_geo.parquet` ذخیره شد. این فایل دارای `999,943` ردیف و `83` ستون است.

</div>

<div dir="rtl" align="right">

## 6. بررسی نهایی خروجی preprocessing

در این بخش فایل نهایی preprocessing یعنی `cleaned_step3_geo.parquet` دوباره خوانده می‌شود.

هدف این مرحله این است که مطمئن شویم خروجی نهایی از نظر تعداد ردیف‌ها، تعداد ستون‌ها، ستون‌های کلیدی و شاخص‌های اصلی آماده‌ی استفاده در مراحل بعدی پروژه است.

</div>

In [36]:
final_data_path = "/content/drive/MyDrive/divar_project/cleaned_step3_geo.parquet"

final_df = pd.read_parquet(final_data_path)

print("Final data loaded.")
print("Shape:", final_df.shape)
print("Columns:", final_df.shape[1])

print("\nMain checks:")
print("rows:", len(final_df))
print("columns:", len(final_df.columns))
print("rows_with_target_price:", final_df["target_price"].notna().sum())
print("valid_for_price_analysis:", final_df["is_valid_for_price_analysis"].sum())
print("valid_geo_for_analysis:", final_df["is_valid_geo_for_analysis"].sum())

print("\nUTM missing values:")
print(final_df[
    [
        "utm_easting",
        "utm_northing",
        "utm_zone_number",
        "utm_zone_letter",
    ]
].isna().sum())

Final data loaded.
Shape: (999943, 85)
Columns: 85

Main checks:
rows: 999943
columns: 85
rows_with_target_price: 917334
valid_for_price_analysis: 877587
valid_geo_for_analysis: 655576

UTM missing values:
utm_easting        344367
utm_northing       344367
utm_zone_number    344367
utm_zone_letter    344367
dtype: int64


<div dir="rtl" align="right">

## 7. جمع‌بندی نهایی preprocessing

در این نوت‌بوک، preprocessing دیتاست املاک دیوار در سه مرحله انجام شد.

در مرحله اول، داده خام خوانده شد، نوع داده‌ها اصلاح شد، تاریخ میلادی به تاریخ شمسی تبدیل شد، بخشی از مقادیر گم‌شده مدیریت شد و ردیف‌هایی که ستون‌های کلیدی ناقص داشتند حذف شدند. خروجی این مرحله با نام `cleaned_step1.parquet` ذخیره شد.

در مرحله دوم، کدهای بنیامین اجرا شد. در این مرحله ستون‌های مربوط به نوع معامله، قیمت مثبت، متراژ مبنا، قیمت واحد، قیمت هدف، flagهای اعتبارسنجی قیمت و flagهای اولیه جغرافیایی ساخته شدند. خروجی این مرحله با نام `cleaned_step2_benyamin_v2.parquet` ذخیره شد.

در مرحله سوم، preprocessing جغرافیایی علی انجام شد. در این مرحله ستون `is_valid_geo_for_analysis` ساخته شد و برای ردیف‌هایی که مختصات معتبر داشتند، latitude و longitude به UTM تبدیل شد. خروجی نهایی این مرحله با نام `cleaned_step3_geo.parquet` ذخیره شد.

خروجی نهایی دارای `999,943` ردیف و `83` ستون است.

تعداد ردیف‌های دارای قیمت هدف:

`917,334`

تعداد ردیف‌های معتبر برای تحلیل قیمت:

`877,587`

تعداد ردیف‌های معتبر برای تحلیل جغرافیایی:

`655,576`

فایل نهایی preprocessing برای استفاده در مراحل بعدی پروژه:

`cleaned_step3_geo.parquet`

</div>